# 🧪 Tool Provider Tests

### 🔁 Step 1: Reset and Seed Tool Provider Configs

In [1]:
from pathlib import Path
import os, sys

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_scores import seed_score_providers
from app.db.seeders.seed_tools import seed_tool_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_tool_providers(session)
    seed_score_providers(session)


Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt GUID: da868a8e-eec2-4427-9cda-ef5872bfd27b
Seeded SystemPrompt GUID: c5c7cc6b-1866-4ccd-8d6b-c31c1cce52b9
Seeded tool configurations successfully.
Seeded score providers successfully.


### 🔍 Step 2: Retrieve Tool Provider ID from DB

In [2]:
from sqlalchemy import select
from app.db.models import ToolProviderConfig

with Session(bind=engine) as session:
    tool_record = session.execute(
        select(ToolProviderConfig).where(ToolProviderConfig.name == "Black Formatter")
    ).scalar_one()
    tool_provider_id = tool_record.id
    print("✅ Found Tool Provider ID:", tool_provider_id)

✅ Found Tool Provider ID: 1


### 🏗️ Step 3: Instantiate Tool Provider from Factory

In [3]:
from app.factories.tool_provider_factory import ToolProviderFactory

tool_provider = ToolProviderFactory.create(tool_provider_id)
print("✅ ToolProvider instantiated:", tool_provider.__class__.__name__)

✅ ToolProvider instantiated: BlackToolProvider


### 🧪 Step 4: Run Tool Provider

In [4]:
result = tool_provider.run(experiment_id='test_exp_001', round=1, target="tests/example.py")
print("✅ Tool Run Result:", result.stdout if result else "No output")

✅ Tool Run Result: 


### 📜 Step 5: Check Tool Provider Logs

In [5]:
from sqlalchemy import text

with engine.connect() as conn:
    rows = conn.execute(
        text("SELECT * FROM tool_provider_log WHERE experiment_id='test_exp_001'")
    ).fetchall()

assert rows, "❌ No tool provider logs found."
print("✅ Logged Tool Runs:")
for row in rows:
    print(row)

✅ Logged Tool Runs:
(1, 'test_exp_001', 1, 1, 'BlackToolProvider', '02dbb08e-405e-4bab-ba32-89020b9ff384', '{"args": [], "kwargs": {"target": "tests/example.py"}}', '', '', 0, 1, None, '2025-05-24T17:30:34.560367+00:00')
